In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
# Create local workspace
mkdir -p /content/KITTI_Project

# Copy code and configs
cp -r /content/drive/MyDrive/KITTI_Project/src /content/KITTI_Project/
cp -r /content/drive/MyDrive/KITTI_Project/configs /content/KITTI_Project/
cp /content/drive/MyDrive/KITTI_Project/requirements.txt /content/KITTI_Project/

# Copy and unzip data locally
mkdir -p /content/KITTI_Project/data
cp /content/drive/MyDrive/KITTI_Project/data/data.zip /content/KITTI_Project/data/
cp /content/drive/MyDrive/KITTI_Project/data/calib.zip /content/KITTI_Project/data/

unzip -q /content/KITTI_Project/data/data.zip -d /content/KITTI_Project/data/
unzip -q /content/KITTI_Project/data/calib.zip -d /content/KITTI_Project/data/

# Create output directory
mkdir -p /content/KITTI_Project/output/exp_0079

In [ ]:
%cd /content/KITTI_Project
!pip install -q -r requirements.txt

In [ ]:
import yaml
import torch
import numpy as np
import os

from src.data.kitti_dataset import KittiDataset
from src.models.gaussian_model import GaussianModel
from src.models.loss_functions import combined_loss
from src.training.trainer import SplatTrainer
from src.training.densifier import Densifier

# Load Configuration
with open("configs/residential_0079.yaml", "r") as f:
    config = yaml.safe_load(f)

print(f"Loaded configuration for experiment: {config['experiment']['name']}")

In [ ]:
# 1. Initialize Dataset
dataset = KittiDataset(config)

# 2. Seed the Point Cloud using the first LiDAR frame
# (KITTI .bin files are stored as float32 in [x, y, z, reflectance] format)
lidar_path = os.path.join(config['data']['lidar_dir'], "0000000000.bin")
scan = np.fromfile(lidar_path, dtype=np.float32).reshape(-1, 4)
pcd_xyz = scan[:, :3] # Extract just the X, Y, Z coordinates

# 3. Initialize the Gaussian Model
model = GaussianModel(sh_degree=config['model']['sh_degree'])
model.create_from_pcd(pcd_xyz=pcd_xyz, device=config['experiment']['device'])

# 4. Initialize the Densifier (Adaptive Density Control)
densifier = Densifier(
    model=model, 
    grad_threshold=config['densification']['grad_threshold'],
    opacity_threshold=config['densification']['opacity_threshold'],
    extent=config['densification']['spatial_extent']
)

# 5. Initialize the Trainer
trainer = SplatTrainer(
    gaussian_model=model,
    dataset=dataset,
    densifier=densifier,
    iterations=config['training']['iterations'],
    device=config['experiment']['device']
)
print("Pipeline fully initialized and ready for training.")

In [ ]:
# Start the optimization loop
trainer.train()

In [ ]:
# 1. Define output paths
local_out_path = os.path.join(config['experiment']['workspace'], "point_cloud.ply")
drive_out_path = f"/content/drive/MyDrive/KITTI_Project/output/exp_0079/point_cloud.ply"

# 2. Save the PLY file locally
model.save_ply(local_out_path)

# 3. Copy back to Google Drive
import shutil
shutil.copy2(local_out_path, drive_out_path)
print(f"Training complete! Point cloud successfully backed up to Google Drive at: {drive_out_path}")